# Moodyjev dijagram — koeficijent trenja

**Poglavlje U10: Realni Bernoulli i gubici**

Ovaj interaktivni prikaz nadopunjuje rad s Moodyjevim dijagramom u poglavlju U10. Mijenjanjem Reynoldsovog broja i relativne hrapavosti cijevi prati se pripadni koeficijent trenja $\lambda$ na klasičnom dijagramu.

## Cilj

Koeficijent trenja $\lambda$ ovisi o Reynoldsovom broju i relativnoj hrapavosti cijevi $\varepsilon/D$. Moodyjev dijagram objedinjuje laminarno područje, prijelaz, gladke cijevi i potpuno hrapavo turbulentno područje na jednom prikazu. Prikaz omogućuje:

1. mijenjanje Reynoldsovog broja $Re$;
2. mijenjanje relativne hrapavosti $\varepsilon/D$;
3. praćenje pripadnog $\lambda$ i režima strujanja.

## Pretpostavke modela

- razvijeno strujanje u kružnoj cijevi konstantnog presjeka;
- jednolika hrapavost stijenke;
- za turbulentno područje koristi se Swamee-Jainova eksplicitna aproksimacija Colebrook-Whiteove jednadžbe;
- za laminarno područje vrijedi egzaktna relacija $\lambda = 64/Re$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

U laminarnom području ($Re < 2300$):

$$\lambda = \frac{64}{Re}.$$

U turbulentnom području ($Re > 4000$) Swamee-Jainova aproksimacija Colebrook-Whiteove jednadžbe:

$$\lambda = \frac{0{,}25}{\left[\log_{10}\!\left(\dfrac{\varepsilon/D}{3{,}7} + \dfrac{5{,}74}{Re^{0{,}9}}\right)\right]^2}.$$

U prijelaznom području ($2300 < Re < 4000$) ponašanje strujanja je nestabilno i $\lambda$ se obično interpolira ili se područje izbjegava u projektiranju.

In [ ]:
def lambda_trenja(Re, eps_D):
    if Re < 2300:
        return 64 / Re, 'laminarno'
    elif Re < 4000:
        # Interpolacija za prijelazno područje
        lam_lam = 64 / 2300
        lam_turb = 0.25 / (np.log10(eps_D/3.7 + 5.74/4000**0.9))**2
        t = (Re - 2300) / (4000 - 2300)
        return lam_lam * (1 - t) + lam_turb * t, 'prijelazno'
    else:
        lam = 0.25 / (np.log10(eps_D/3.7 + 5.74/Re**0.9))**2
        return lam, 'turbulentno'

## Interaktivni prikaz

Klizačima u nastavku biraju se Reynoldsov broj i relativna hrapavost. Prikaz je Moodyjev dijagram u logaritamskim koordinatama s istaknutom točkom koja odgovara odabranim vrijednostima.

In [ ]:
def moody_prikaz(log_Re, eps_D):
    Re = 10**log_Re
    lam, rezim = lambda_trenja(Re, eps_D)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Laminarna linija
    Re_lam = np.logspace(2.5, np.log10(2300), 50)
    ax.loglog(Re_lam, 64/Re_lam, color='#1565c0', lw=2,
              label='laminarno  $\\lambda = 64/Re$')

    # Turbulentne krivulje za nekoliko ε/D
    eps_D_niz = [1e-6, 1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]
    Re_turb = np.logspace(np.log10(4000), 8, 100)
    for e in eps_D_niz:
        lam_kr = 0.25 / (np.log10(e/3.7 + 5.74/Re_turb**0.9))**2
        ax.loglog(Re_turb, lam_kr, color='#888', lw=0.8, alpha=0.6)
        ax.text(Re_turb[-1] * 1.1, lam_kr[-1], f'{e:.0e}',
                 fontsize=8, color='#666', va='center')

    # Krivulja za odabrani ε/D
    lam_odab = 0.25 / (np.log10(eps_D/3.7 + 5.74/Re_turb**0.9))**2
    ax.loglog(Re_turb, lam_odab, color='#2e7d32', lw=2.2,
              label=f'odabrano $\\varepsilon/D$ = {eps_D:.1e}')

    # Prijelazno područje
    ax.axvspan(2300, 4000, color='#fff3e0', alpha=0.5)
    ax.text(3000, 0.005, 'prijelazno', rotation=90, fontsize=9,
             ha='center', color='#e65100')

    # Točka
    ax.scatter([Re], [lam], color='#c62828', s=140,
                zorder=5, marker='o',
                label=f'$Re$ = {Re:.1e},  $\\lambda$ = {lam:.4f}')
    ax.annotate(f'  {rezim}', xy=(Re, lam),
                 fontsize=10, color='#c62828', va='center')

    ax.set_xlabel('Reynoldsov broj  $Re$')
    ax.set_ylabel('koeficijent trenja  $\\lambda$')
    ax.set_xlim(500, 1e8)
    ax.set_ylim(0.005, 0.1)
    ax.grid(which='both', ls=':', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_title(f'Moodyjev dijagram   |   režim: {rezim}',
                  fontsize=11)

    plt.tight_layout()
    plt.show()


interact(
    moody_prikaz,
    log_Re=FloatSlider(min=2.5, max=8, step=0.1, value=5,
                        description='log$_{10}$ Re',
                        readout_format='.2f',
                        layout=Layout(width='420px')),
    eps_D=FloatSlider(min=1e-6, max=5e-2, step=1e-5, value=1e-4,
                       description='$\\varepsilon/D$',
                       readout_format='.1e',
                       layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Potpuno hrapavo područje.** Pri visokim Reynoldsovim brojevima krivulje $\lambda(Re)$ za zadanu hrapavost prelaze u horizontalne pravce. U kojem trenutku $Re$ više ne utječe na $\lambda$? Što to fizikalno znači?

2. **Hidraulički gladke cijevi.** Za vrlo malu hrapavost $\varepsilon/D < 10^{-5}$, kako se $\lambda$ ponaša s porastom $Re$ u turbulentnom području? Zašto se cijevi te klase nazivaju 'hidraulički gladke'?

3. **Prijelazno područje.** U području $2300 < Re < 4000$ $\lambda$ je teško predvidljiv. Zašto se u inženjerskoj praksi izbjegava projektirati radne točke u tom području?

4. **Inverzni zadatak.** Za $\lambda \approx 0{,}025$, koje kombinacije $Re$ i $\varepsilon/D$ ga daju? Postoji li više rješenja i što govore o različitim radnim režimima cijevi?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira Moodyjev dijagram iz poglavlja U10 — klasično grafičko rješenje Colebrook-Whiteove jednadžbe koje objedinjuje sve režime strujanja u cijevi. Iako se danas $\lambda$ često računa eksplicitnim aproksimacijama (Swamee-Jain, Haaland), Moodyjev dijagram ostaje neprocjenjiv kao alat za razumijevanje međusobne ovisnosti hrapavosti, Reynoldsovog broja i koeficijenta trenja.